# カメラ画角の走行シミュレーション

車体にカメラを載せる前に、画角の中で赤がどう動くかを見る。カメラもモーターも使わない。

Logitech Brio 100 の対角視野は製品仕様の 58° とする。取り込みの既定が 1280×720 なので、画面は 16:9 である。この比に分配すると水平は約 52°、垂直は約 30° になる。

車輪の停止・旋回・前進は `flycar/behavior.py` と同じである。球の投影と走行は `flycar/fov.py`、俯瞰・カメラ・ニューロンの図は `flycar/view.py` にある。ノートを増やすときも、新しい Python アプリからも、同じ関数を呼ぶ。このノートは初期条件を書いて、走らせて、見る。

カウントは、画面に写った球が左右の半画面を水平にどれだけ占めるかである。縦にはみ出した分は幅を減らさない。20 ms ごとに絵を測り、そのカウントを次の 20 ms の膜へ入れ続ける。膜の刻みは 1 ms のままである。

固定フレームの pytest（`tests/test_behavior.py`）はこの幾何を通さない。下の基準場面は `tests/test_fov.py` が確認する。球の位置を変えた場面は `notebooks/fov_scenes.ipynb` にある。


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "flycar" / "behavior.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flycar.fov import Blob, Scene, simulate
from flycar.view import neuron_figure, show_world


## 初期条件

球は地面に置く。中心の高さは半径。俯角は下向きが正。車は原点から +x を向いて出る。対角 58° と 16:9 は `Scene` の既定である。

この基準場面では、赤い球を離れた右の画角に置き、青い球は左前方、最初の画角の外に置く。


In [ ]:
scene = Scene(
    cam_h=0.15,
    pitch_deg=12.0,
    red=Blob((1.30, -0.40), 0.12),
    blue=Blob((0.25, 0.90), 0.10),
    run_s=8.0,
)
print(scene)


## シミュレーション


In [ ]:
run = simulate(scene)
print(run)


## 俯瞰とカメラ

左は上から見た車、球、水平画角である。右はカメラに写っている球で、水平中央に白い線を引いてある。再生すると、車が動くたびの絵になる。表示範囲は車の軌跡と球に合わせる。

この 12° では、赤は画面の上半分から中央寄りまで下がって止まる。水平方向は中央へ寄る。右へ回りすぎて 0.3 秒ごろ中央を左へ越え、0.5 秒には戻り、0.8 秒にはほぼ中央に乗る。


In [ ]:
show_world(run)


## ニューロンの入力と出力

入力は細胞への電流、出力は車輪へ渡している発火率である。赤は DNa02、青は DNp01。実線が左、破線が右。電流 1 がスパイクの閾値である。灰色は停止している区間である。

この基準場面では青は画面の外なので、DNp01 の電流は 0 のままである。右の赤で右の DNa02 が先に発火し、中央を越えたあいだだけ左も発火して戻す。中央に乗ったあとは左右の電流がいったん閾値を下回り、近づくと両方の電流が閾値を超える。


In [ ]:
neuron_figure(run)
